In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Read one config row and publish its fields as task values for downstream tasks
# (warehouse SQL tasks + cluster notebooks read {{tasks.get_task.values.<field>}}).
task_id = widget("task_id")
row = store.get_task(task_id)
if not row:
    raise ValueError(f"No control row for task_id={task_id}")


def as_str(v):
    # Booleans -> 'true'/'false' to match reused SQL checks (:use_remote_query = 'true') and
    # condition tasks; None -> '' (optional sink_cluster_cols / src_catalog).
    if v is None:
        return ""
    if isinstance(v, bool):
        return "true" if v else "false"
    return str(v)


for key, value in row.items():
    dbutils.jobs.taskValues.set(key=key, value=as_str(value))
print(row)